# 06 Pooling And Representation

Reviewer-facing pooling sensitivity and representation-evidence status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Scope And Contracts

This notebook has one executable analysis with an implemented script: pooling sensitivity from frozen OOF slice probabilities. Representation probing remains a separate blocked task until embeddings/probing runners exist; this notebook records that status explicitly rather than inferring branch separation from pooling.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

THRESHOLD = 0.5
RUN_POOLING_SENSITIVITY = True
RUN_REPRESENTATION_PROBING = False
OOF_STEM = 'oof_final_ema'

revision_root = Path('reports/revision')
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
record_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
slice_path = revision_root / 'predictions' / f'{OOF_STEM}_slice_predictions.npz'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'

required_inputs = {
    'oof_final_ema_freeze_manifest': freeze_path,
    'frozen_final_ema_predictions': record_path,
    'frozen_final_ema_slice_predictions': slice_path,
    'calibration_ci_final_ema': calibration_path,
}
for name, path in required_inputs.items():
    print(f'{name:34s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 02/03 final_ema artifacts are required before notebook 06: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before pooling sensitivity.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 06 requires checkpoint_kind=final_ema in the freeze manifest.')

frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
for artifact_path in [record_path, slice_path]:
    rel = artifact_path.as_posix()
    if rel not in frozen_artifacts:
        raise ValueError(f'Freeze manifest does not include {rel}')
    current_sha = sha256_file(artifact_path)
    if current_sha != frozen_artifacts[rel]['sha256']:
        raise RuntimeError(f'Frozen artifact checksum changed: {rel}')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'oof_stem': OOF_STEM,
    'threshold': THRESHOLD,
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'checkpoint_kind': freeze.get('checkpoint_kind'),
    'allowed_execution': {
        'pooling_sensitivity': RUN_POOLING_SENSITIVITY,
        'representation_probing': RUN_REPRESENTATION_PROBING,
        'model_training': False,
        'model_inference': False,
    },
}
save_json(revision_root / 'manifests' / 'pooling_representation_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'pooling_representation_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 06 to the tracked revision plan and claim map. It separates supported pooling analysis from still-blocked representation evidence.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_POOL', 'EXP_REPR'])]
relevant_claims = claims[claims['claim_id'].isin(['C04', 'C05'])]
relevant_tasks = tasks[tasks['id'].isin(['B1', 'B3'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'pooling_representation_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'pooling_representation_plan_alignment.json')


## Run Pooling Sensitivity

This uses only checksum-verified frozen OOF slice probabilities. It performs no model inference, no training, and no threshold tuning.


In [ ]:
pooling_command = (
    f'python -u scripts/revision/07_pooling_sensitivity.py '
    f'--freeze-manifest {freeze_path} '
    f'--record-file {record_path} '
    f'--slice-file {slice_path} '
    f'--threshold {THRESHOLD}'
)
if RUN_POOLING_SENSITIVITY:
    run(pooling_command, log_path='reports/revision/logs/oof_final_ema_pooling_sensitivity.log')
else:
    print('Pooling sensitivity disabled. Planned command:', pooling_command)


## Inspect Pooling Results And Decide Q=3 Wording

The decision artifact records whether Q=3 is best, competitive, or should be described only as the frozen/default aggregation setting.


In [ ]:
pooling_csv = revision_root / 'metrics' / 'pooling_sensitivity.csv'
pooling_json = revision_root / 'metrics' / 'pooling_sensitivity.json'
if not pooling_csv.exists() or not pooling_json.exists():
    raise FileNotFoundError('Pooling sensitivity outputs are missing; run the previous cell first.')

pooling_df = pd.read_csv(pooling_csv)
expected_methods = {'mean', 'power_mean_q2', 'power_mean_q3', 'power_mean_q4', 'power_mean_q8', 'max'}
observed_methods = set(pooling_df['pooling'].astype(str))
missing_methods = sorted(expected_methods - observed_methods)
if missing_methods:
    raise ValueError('Pooling sensitivity is incomplete; missing methods: ' + ', '.join(missing_methods))

metric_priority = ['roc_auc_macro', 'pr_auc_macro', 'f1_macro', 'recall_macro', 'specificity_macro']
q3 = pooling_df[pooling_df['pooling'] == 'power_mean_q3'].iloc[0].to_dict()
rank_summary = {}
for metric in metric_priority:
    ordered = pooling_df.sort_values(metric, ascending=False).reset_index(drop=True)
    rank = int(ordered.index[ordered['pooling'] == 'power_mean_q3'][0] + 1)
    best = ordered.iloc[0].to_dict()
    rank_summary[metric] = {
        'q3_rank': rank,
        'q3_value': float(q3[metric]),
        'best_pooling': best['pooling'],
        'best_value': float(best[metric]),
    }

q3_best_metrics = [metric for metric, row in rank_summary.items() if row['q3_rank'] == 1]
if q3_best_metrics:
    q3_decision = 'q3_supported_for_metrics_' + '_'.join(q3_best_metrics)
    safe_wording = 'Q=3 is supported for the listed metrics but should still be reported with the full sensitivity table.'
else:
    q3_decision = 'q3_not_best_present_as_frozen_default_or_tradeoff'
    safe_wording = 'Do not claim Q=3 is optimal; present it as the frozen/default aggregation and report the better pooling alternatives.'

decision = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_pooling_csv': str(pooling_csv),
    'source_pooling_json': str(pooling_json),
    'source_pooling_csv_sha256': sha256_file(pooling_csv),
    'threshold': THRESHOLD,
    'expected_methods': sorted(expected_methods),
    'rank_summary': rank_summary,
    'q3_decision': q3_decision,
    'safe_wording': safe_wording,
}
save_json(revision_root / 'metrics' / 'pooling_decision_summary.json', decision)
pooling_table = revision_root / 'tables' / 'table_pooling_sensitivity.csv'
pooling_df.to_csv(pooling_table, index=False)
print('Wrote:', revision_root / 'metrics' / 'pooling_decision_summary.json')
print('Wrote:', pooling_table)
display(pooling_df)
print('Q=3 decision:', q3_decision)
print('Safe wording:', safe_wording)


## Representation Evidence Status

Representation probing/UMAP/CKA are not implemented in this branch. This cell writes an explicit blocked ledger so the response letter does not overstate morphology-rhythm separation.


In [ ]:
representation_outputs = {
    'probing_summary': revision_root / 'metrics' / 'probing_summary.csv',
    'representation_umap': revision_root / 'figures' / 'representation_umap.png',
}
representation_rows = []
for name, path in representation_outputs.items():
    representation_rows.append({
        'evidence_name': name,
        'path': str(path),
        'exists': path.exists(),
        'status': 'complete' if path.exists() else 'blocked_runner_tbd',
        'blocker': '' if path.exists() else 'No implemented representation probing or visualization runner is available in scripts/revision.',
        'safe_wording': 'Use artifact-specific evidence only.' if path.exists() else 'Do not claim demonstrated morphology-rhythm separation; state this remains future/limited evidence.',
    })

repr_df = pd.DataFrame(representation_rows)
repr_table = revision_root / 'tables' / 'table_representation_evidence_status.csv'
repr_json = revision_root / 'metrics' / 'representation_evidence_status.json'
repr_df.to_csv(repr_table, index=False)
save_json(repr_json, {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'run_representation_probing': RUN_REPRESENTATION_PROBING,
    'status': 'complete' if all(row['exists'] for row in representation_rows) else 'blocked_runner_tbd',
    'rows': representation_rows,
})
print('Wrote:', repr_table)
print('Wrote:', repr_json)
display(repr_df)


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/pooling_representation_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/pooling_representation_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'pooling_representation_input_contract.json',
    revision_root / 'manifests' / 'pooling_representation_plan_alignment.json',
    revision_root / 'metrics' / 'pooling_sensitivity.csv',
    revision_root / 'metrics' / 'pooling_sensitivity.json',
    revision_root / 'metrics' / 'pooling_decision_summary.json',
    revision_root / 'tables' / 'table_pooling_sensitivity.csv',
    revision_root / 'metrics' / 'representation_evidence_status.json',
    revision_root / 'tables' / 'table_representation_evidence_status.csv',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
print('Notebook 06 complete. Pooling sensitivity can support/revise Q=3 wording; representation separation remains blocked unless probing/UMAP artifacts exist.')
